In [1]:
from mri_dataset import MRIPatient25DDataset

In [5]:
import numpy as np
import torch
from torch.utils.data import Dataset
from ionifti import get_sorted_nifti_acquisitions
class PatientMultimodalDataset(Dataset):
    """
    1 item = 1 paciente
    Devuelve:
      x_img: (C,H,W)  (C=3*k si usas pre/early/late por k slices)
      x_clin: (d,)
      y: (1,)
      pid: str
    """
    def __init__(self, df, clin_cols, k=3, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.clin_cols = clin_cols
        self.k = k
        self.transform = transform

        # filtra filas inválidas
        self.df = self.df.dropna(subset=["pCR", "mask_start", "mask_end"])
        self.df = self.df.dropna(subset=clin_cols)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["pid"]
        y = torch.tensor([float(row["pCR"])], dtype=torch.float32)

        # ---- clínica ----
        x_clin = torch.tensor(row[self.clin_cols].values.astype(np.float32))

        # ---- imagen (9 canales si k=3) ----
        pmask = int((row["mask_start"] + row["mask_end"]) // 2)

        d = get_sorted_nifti_acquisitions(pid)  # lista de 3 vols: [pre, early, late] (Z,H,W)
        if d is None or len(d) < 3:
            raise RuntimeError(f"Missing DCE phases for {pid}")

        nz = d[0].shape[0]
        half = self.k // 2
        zs = [max(0, min(pmask + off, nz - 1)) for off in range(-half, half + 1)]

        chans = []
        for z in zs:
            chans.append(d[0][z])  # pre
            chans.append(d[1][z])  # early
            chans.append(d[2][z])  # late

        x_img = np.stack(chans, axis=0).astype(np.float32)  # (3*k,H,W)
        x_img = torch.from_numpy(x_img)

        if self.transform:
            x_img = self.transform(x_img)

        return x_img, x_clin, y, pid


In [4]:
#model.forward_features(x) → embedding para multimodal
from mri_cnn_model import A1_MRI_CNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = A1_MRI_CNN(in_ch=9).to(device)
model.load_state_dict(torch.load("weights.pt", map_location=device))
model.eval()


C:\Users\Ivón\AppData\Local\Temp\ipykernel_2824\2824269447.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("weights.pt", map_location=de

A1_MRI_CNN(
  (features): Sequential(
    (0): Conv2d(9, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU(inplace=True)
  )
  (gap): AdaptiveAvgPool2d(output_size=(1, 1))
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=1, bi

In [ ]:
import numpy as np
from collections import defaultdict
import torch

@torch.no_grad()
def extract_patient_embeddings(model, dl, device, agg="mean"):
    model.eval()
    pid2emb = defaultdict(list)
    pid2y = {}

    for xb, yb, pid in dl:
        xb = xb.to(device)
        yb = yb.view(-1).cpu().numpy()
        pid = list(pid)

        emb = model.forward_features(xb).cpu().numpy()  # (B,256)

        for p, e, yy in zip(pid, emb, yb):
            pid2emb[p].append(e)
            pid2y[p] = yy

    pids = sorted(pid2emb.keys())
    if agg == "mean":
        X = np.stack([np.mean(pid2emb[p], axis=0) for p in pids])
    else:
        X = np.stack([np.max(pid2emb[p], axis=0) for p in pids])

    y = np.array([pid2y[p] for p in pids]).astype(int)
    return X, y, pids


In [ ]:

Xtr, ytr, pids_tr = extract_patient_embeddings(model, train_loader, device)
np.savez("cnn_embed_train.npz", X=Xtr, y=ytr, pid=np.array(pids_tr))


In [ ]:
Xtr, ytr, pid_tr = extract_patient_embeddings(model, train_loader, device, agg="mean")
Xva, yva, pid_va = extract_patient_embeddings(model, val_loader, device, agg="mean")
Xte, yte, pid_te = extract_patient_embeddings(model, test_loader, device, agg="mean")

import numpy as np
np.savez("emb_train.npz", X=Xtr, y=ytr, pid=np.array(pid_tr))
np.savez("emb_val.npz",   X=Xva, y=yva, pid=np.array(pid_va))
np.savez("emb_test.npz",  X=Xte, y=yte, pid=np.array(pid_te))
